# **Multi-GPU Spherical Particle-Mesh Lightcone**

<a href="https://colab.research.google.com/github/DifferentiableUniverseInitiative/JaxPM/blob/main/docs/notebooks/04-MultiGPU_PM_Spherical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook runs an **end-to-end spherical (lightcone) simulation distributed across
multiple devices**. It mirrors the single-GPU spherical pipeline of
[`08-convergence-vs-glass`](08-convergence-vs-glass.ipynb), but every stage is sharded
over a 2×4 device mesh:

1. **Distributed initial conditions** and first-order LPT.
2. **FastPM** symplectic kick/drift operators (`symplectic_fpm_ode`) integrated with
   diffrax's `SemiImplicitEuler` at a fixed step.
3. **`SaveAt`** paints one **distributed HEALPix density shell** per saved scale factor,
   building a full-sky lightcone on the fly.
4. The **spherical map itself is sharded** across devices (shown explicitly below).
5. **Overdensity shells** and their angular power spectra $C_\ell$ via `healpy.anafast`.

No adaptive ODE solver (no `Tsit5` / `Dopri5`) and no deprecated painting helpers are
used — the simulation relies entirely on the modern `paint` / spherical API.

In [ ]:
# Install JaxPM and healpy. Runs only when the packages are missing (e.g. on
# Colab); an existing local install is left untouched.
try:
    import jaxpm  # noqa: F401
    import healpy  # noqa: F401
except ImportError:
    %pip install -q "jaxpm @ git+https://github.com/DifferentiableUniverseInitiative/JaxPM.git"
    %pip install -q healpy

> **Note**: This notebook requires 8 devices (GPU or TPU).\
> If you're running on CPU or don't have access to 8 devices,\
> you can simulate multiple devices by adding the following code at the start **BEFORE IMPORTING JAX**:

```python
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=8"
```

**Recommended only for debugging**. If used, you should lower the `mesh_size` / `nside`.

In [ ]:
import os

os.environ["EQX_ON_ERROR"] = "nan"
import jax
import jax.numpy as jnp
import jax_cosmo as jc

import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
from functools import partial

from diffrax import ODETerm, SaveAt, diffeqsolve, SemiImplicitEuler, ConstantStepSize

from jax.experimental.multihost_utils import process_allgather
from jax.sharding import PartitionSpec as P, NamedSharding, AxisType
from jax.debug import visualize_array_sharding

from jaxpm.pm import linear_field, lpt
from jaxpm.ode import symplectic_fpm_ode
from jaxpm.distributed import uniform_particles
from jaxpm.lensing import spherical_density_fn
from jaxpm.spherical import paint_particles_spherical
from jaxpm.kernels import interpolate_power_spectrum

jax.config.update("jax_enable_x64", True)

In [ ]:
assert jax.device_count() >= 8, (
    "This notebook requires a TPU or GPU runtime with 8 devices"
)

### Device mesh and sharding

We arrange the 8 devices in a **2×4 mesh** and shard every field along the `("x", "y")`
axes. The same `sharding` object is threaded through the initial conditions, the LPT
displacements, the FastPM force computation, and the spherical painter, so data stays
distributed end to end and is never gathered onto a single device.

In [ ]:
all_gather = partial(process_allgather, tiled=True)

pdims = (2, 4)
# Build the mesh with *Auto* axis types: jaxDecomp's distributed FFT and halo-exchange
# primitives rely on JAX's automatic (compiler-managed) sharding (see the note below).
mesh = jax.make_mesh(
    pdims, axis_names=("x", "y"), axis_types=(AxisType.Auto, AxisType.Auto)
)
sharding = NamedSharding(mesh, P("x", "y"))

> **Auto vs Explicit axis types** — the mesh is built with
`axis_types=(AxisType.Auto, AxisType.Auto)`. jaxDecomp's distributed FFT and
halo-exchange primitives (used under the hood by every sharded `linear_field` / `lpt` /
`paint` / `symplectic_fpm_ode` / spherical-painting call) rely on JAX's **automatic**,
compiler-managed sharding. An *Explicit*-axis mesh — the default of `jax.make_mesh(...)`
without `axis_types` — would break these primitives. See
[jaxDecomp · Auto vs Explicit axis types](https://jaxdecomp.readthedocs.io/en/latest/07-xla_sharding_configuration.html#auto-vs-explicit-axis-types)
for the full explanation.

## Configuration

`mesh_size` and `nside` are the only knobs to scale up: increase them for a cluster run,
decrease them when faking devices on CPU for debugging.

In [ ]:
mesh_size = 512  # particle-mesh resolution (mesh_size^3), sharded over the device mesh
nside = 512      # HEALPix resolution of the distributed spherical shells / maps
# For a larger cluster run, scale both together, e.g. mesh_size = 1024, nside = 1024.
n_shells = 40    # number of spherical density shells along the lightcone
z_source = 0.8   # furthest source redshift (sets the comoving box size)
seed = 42

# FastPM time stepping in scale factor. A fixed step is required: the FastPM kick/drift
# factors are written per unit dt0, so they only reproduce FastPM when every diffrax step
# equals dt0 (ConstantStepSize). Pick dt0 so it evenly divides (t1 - t0).
t0, t1, dt0 = 0.1, 1.0, 0.05
halo_size = 64   # halo-exchange width for the distributed force computation

# Observer at the box centre -> the chi(z_source) sphere is inscribed in the box,
# giving a full-sky lightcone with no periodic replication.
observer_in_box = jnp.array([0.5, 0.5, 0.5])

mesh_shape = (mesh_size,) * 3
npix = hp.nside2npix(nside)

## Cosmology and geometry

The box is sized as `2 * chi(z_source)` so the furthest source sphere reaches exactly the
box face from the centred observer. Shells are uniform in comoving distance and converted
to scale factors with `jax_cosmo`.

In [ ]:
cosmo = jc.Planck18()

# box = 2 * chi(z_source) so the source sphere reaches exactly the box face.
r_max = jc.background.radial_comoving_distance(cosmo, jc.utils.z2a(z_source)).squeeze()
box_size = tuple(float(2.0 * r_max) for _ in range(3))
observer_position = observer_in_box * jnp.array(box_size)

# Uniform shells in comoving distance.
r_edges = jnp.linspace(0.0, r_max, n_shells + 1)
r_center = 0.5 * (r_edges[1:] + r_edges[:-1])
a_center = jc.background.a_of_chi(cosmo, r_center)
d_R = float(r_max / n_shells)

print(f"box size       : {box_size[0]:.0f} Mpc/h")
print(f"chi(z={z_source}) : {float(r_max):.0f} Mpc/h")
print(f"shell width    : {d_R:.1f} Mpc/h over {n_shells} shells")
print(
    f"shell redshifts: z = {float(1 / a_center.max() - 1):.2f} .. {float(1 / a_center.min() - 1):.2f}"
)

## 1. Distributed density lightcone

LPT initial conditions are evolved with the **FastPM** solver. At each saved scale factor
the `SaveAt` callback paints the particles that fall inside the corresponding shell onto a
**distributed** HEALPix map via `spherical_density_fn` (which forwards `sharding` to the
spherical painter).

Because we integrate with the FastPM kick/drift operators, the state is `(displacement,
velocity)` and the operators are ordered `(drift, kick)` so the kick sees the freshly
drifted position. The FastPM scheme also needs an initial half-kick (`first_kick`) to
stagger the velocities, applied once before the integration.

In [ ]:
# Linear power spectrum -> distributed Gaussian initial conditions.
k = jnp.logspace(-3, 1, 256)
pk = jc.power.linear_matter_power(cosmo, k)
pk_fn = lambda x: interpolate_power_spectrum(x, k, pk, sharding)
ic = linear_field(mesh_shape, box_size, pk_fn, seed=jax.random.PRNGKey(seed), sharding=sharding)

# First-order LPT at the starting scale factor.
dx, p, _ = lpt(cosmo, ic, a=t0, order=1, halo_size=halo_size, sharding=sharding)

# FastPM symplectic kick/drift operators (positions stored as displacements).
drift, kick, first_kick = symplectic_fpm_ode(
    mesh_shape, dt0, cosmo, initial_particles="uniform",
    halo_size=halo_size, sharding=sharding,
)
# FastPM initial half-kick to stagger the velocities.
p = p + dt0 * first_kick(t0, dx, cosmo)

# Paint one distributed HEALPix shell per saved scale factor (saved far -> near in a).
# The state is (displacement, velocity), so we hand y[0] (the displacement) to the painter.
paint_shell = spherical_density_fn(
    mesh_shape, box_size, cosmo, nside, observer_position, d_R,
    method="NGP", sharding=sharding,
)
saveat = SaveAt(ts=a_center[::-1], fn=lambda t, y, args: paint_shell(t, y[0], args))

sol = diffeqsolve(
    (ODETerm(drift), ODETerm(kick)),
    SemiImplicitEuler(),
    t0=t0,
    t1=t1,
    dt0=dt0,
    y0=(dx, p),
    args=cosmo,
    saveat=saveat,
    stepsize_controller=ConstantStepSize(),
)

# Order the shells near -> far so they align with r_center / a_center.
density_planes = sol.ys[::-1]
print("density planes:", density_planes.shape)

## 2. The spherical map is distributed

Each shell is painted as a HEALPix map **sharded across the devices**: the pixels are split
along the first mesh axis (and replicated along the second), so no device ever holds the
full sphere during the simulation. The diffrax `SaveAt` buffer gathers its stacked outputs
for convenience, so to expose the sharding we paint a single shell on its own (with the
particle positions kept explicitly sharded) and inspect the result. The
`visualize_array_sharding` output shows the sphere's pixels divided between the devices.

In [ ]:
# Paint one shell directly, keeping the particle positions sharded, so the resulting
# HEALPix map stays distributed (the diffrax save buffer above gathers its outputs).
abs_positions = uniform_particles(mesh_shape, sharding=sharding) + dx
shell_idx0 = n_shells // 2
shell_map = paint_particles_spherical(
    abs_positions,
    nside=nside,
    observer_position=observer_position,
    R_min=float(r_edges[shell_idx0]),
    R_max=float(r_edges[shell_idx0 + 1]),
    box_size=jnp.array(box_size),
    mesh_shape=mesh_shape,
    method="NGP",
    sharding=sharding,
)
shell_map.block_until_ready()

print("shell map shape    :", shell_map.shape)
print("shell map sharding :", shell_map.sharding.spec)
print("fully replicated   :", shell_map.sharding.is_fully_replicated)
visualize_array_sharding(shell_map)

In [ ]:
# Mollweide view of a gathered density shell from the lightcone.
shell_idx = n_shells // 2
plane = np.asarray(jax.device_get(density_planes[shell_idx]))
z_shell = 1.0 / float(a_center[shell_idx]) - 1.0
hp.mollview(
    np.log10(plane + 1e-6),
    title=f"density shell {shell_idx}  (z = {z_shell:.2f})",
    cmap="magma",
)
plt.show()

## 3. Overdensity shells and angular power spectra

We convert a few representative shells (near, middle, far) to overdensity
$\delta = \rho / \bar{\rho} - 1$ and measure their angular power spectra with
`healpy.anafast`. More distant shells subtend more volume and are less evolved, so their
clustering power drops — the expected lightcone trend.

In [ ]:
lmax = 2 * nside - 1
ell = np.arange(lmax + 1)
ellp = ell[2:]
coef = ellp * (ellp + 1) / (2.0 * np.pi)

idxs = [0, n_shells // 2, n_shells - 1]  # near, middle, far

plt.figure(figsize=(7, 5))
for i in idxs:
    plane = np.asarray(jax.device_get(density_planes[i]))
    delta = plane / plane.mean() - 1.0
    cl = hp.anafast(delta, lmax=lmax)[2:]
    z = 1.0 / float(a_center[i]) - 1.0
    plt.loglog(ellp, coef * cl, label=f"shell {i}  (z = {z:.2f})")

plt.xlabel(r"$\ell$")
plt.ylabel(r"$\ell(\ell+1)\,C_\ell^{\delta\delta}/2\pi$")
plt.title("Angular power spectra of the distributed overdensity shells")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()